# 07 — Gerar e conferir o artigo em LaTeX

Este notebook lê os resultados dos notebooks analíticos, gera `artigo.tex` e `referencias.bib`, compila o PDF (se o LaTeX estiver instalado) e confere extensão e referências.

O texto é organizado por três perguntas (P1–P3). Cada pergunta dá nome a uma parte dos Métodos e dos Resultados, e a conclusão responde a cada uma. O artigo tem **quatro figuras**, todas geradas por notebooks:

| Figura | Notebook | O que mostra |
|---|---|---|
| 1 | `08_metodo_setor_PA_dijkstra` | como cada setor censitário é ligado a um PA (ruas + Dijkstra) |
| 2 | `09_escala_L1_L2` | P1 e P2: gráficos log-log de L1 e L2 e o teste de embaralhamento |
| 3 | `11_gini_distancia` | P3 (quanto): distâncias e curva de Lorenz/Gini dentro das capitais |
| 4 | `06_figura_final` | P3 (quem): o P90 em Fortaleza, % de moradores além do P90 por renda e densidade nas 26 capitais, e mapa de Fortaleza |

**Escolhas registradas:** distância mínima total pelas ligações e ruas; sem percolação; o L1 também é refeito com o aglomerado urbano do IBGE (notebook 10); modelo simples de renda sem efeitos fixos. Autor e afiliação: Lia Sucupira, Universidade de Fortaleza.

Todos os números do texto vêm dos CSVs; nada é digitado à mão. Se alterar o `.tex` diretamente, as mudanças são perdidas ao reexecutar este notebook.

In [1]:
from pathlib import Path
import json
import re
import shutil
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PLANO.md").exists())
ARTIGO = RAIZ / "artigo"
ARTIGO.mkdir(exist_ok=True)
BASE = RAIZ / "dados" / "processados" / "analises"

AUTOR = "Lia Sucupira"
AFILIACAO = "Universidade de Fortaleza"

l1 = pd.read_csv(BASE/"L1/resultados_L1.csv").set_index("versao")
l1agl = pd.read_csv(BASE/"L1/resultados_L1_aglomerado.csv").set_index(["versao", "definicao_de_cidade"])
l2 = pd.read_csv(BASE/"L2/resultados_L2.csv").set_index("versao")
nulo = pd.read_csv(BASE/"L2/embaralhamento_L2.csv").set_index(["versao", "embaralhamento"])
renda = pd.read_csv(BASE/"renda/resultados_renda.csv").set_index("versao")
distancias = pd.read_csv(BASE/"distancias/resumo_capitais.csv")
num = pd.read_csv(BASE/"distancias/numeros_figura_principal.csv").set_index("nome").valor
cap = pd.read_csv(BASE/"L1/capitais_sus.csv")
gini_rho = pd.read_csv(BASE/"distancias/gini_vs_PAs.csv", index_col=0)
gini_cap = pd.read_csv(BASE/"distancias/gini_capitais.csv").set_index("nome")
for f in ["metodo/metodo_setor_pa", "escala/escala_L1_L2", "final/figura_principal", "gini/gini_PA"]:
    assert (RAIZ/f"figuras/{f}.pdf").exists(), f
TEM_LATEX = bool(shutil.which("pdflatex") and shutil.which("bibtex") and shutil.which("latexmk"))
print("LaTeX instalado:", TEM_LATEX)
print(cap.columns.tolist())


LaTeX instalado: True
['municipio', 'co_ibge7', 'populacao_2022', 'n_pa', 'n_pa_sus', 'n_pa_24h', 'n_nome_upa', 'n_com_coord', 'n_unidades', 'versao', 'log10_residuo']


## 1. Referências verificadas nos arquivos do projeto

As referências teóricas foram conferidas nos PDFs locais. As fontes de dados CNES/OpenDataSUS, IBGE e OpenStreetMap/Geofabrik são identificadas nos métodos.

O artigo usa a L3 de forma descritiva, sem estimar expoente de cauda. O modelo de renda é o modelo conjunto simples escolhido pela autora. Não comparamos localizações reais com uma solução otimizada.

In [2]:
bibliografia = r"""@book{bettencourt2021,
  author = {Bettencourt, Lu{\'i}s M. A.},
  title = {Introduction to Urban Science: Evidence and Theory of Cities as Complex Systems},
  publisher = {The MIT Press},
  year = {2021}
}
@article{gastner2006,
  author = {Gastner, Michael T. and Newman, M. E. J.},
  title = {Optimal design of spatial distribution networks},
  journal = {Physical Review E},
  volume = {74},
  pages = {016117},
  year = {2006},
  doi = {10.1103/PhysRevE.74.016117}
}
@article{xu2020,
  author = {Xu, Yanyan and Olmos, Luis E. and Abbar, Sofiane and Gonz{\'a}lez, Marta C.},
  title = {Deconstructing laws of accessibility and facility distribution in cities},
  journal = {Science Advances},
  volume = {6},
  pages = {eabb4112},
  year = {2020},
  doi = {10.1126/sciadv.abb4112}
}
@article{melo2014,
  author = {Melo, Hygor Piaget M. and Moreira, Andr{\'e} A. and Batista, {\'E}lcio and Makse, Hern{\'a}n A. and Andrade, Jos{\'e} S.},
  title = {Statistical Signs of Social Influence on Suicides},
  journal = {Scientific Reports},
  volume = {4},
  pages = {6239},
  year = {2014},
  doi = {10.1038/srep06239}
}
@article{newman2005,
  author = {Newman, M. E. J.},
  title = {Power laws, Pareto distributions and {Zipf}'s law},
  journal = {Contemporary Physics},
  volume = {46},
  pages = {323--351},
  year = {2005}
}
@article{maia2023,
  author = {Maia, Gabriel and Ponte, Caio and Caminha, Carlos and Furtado, Lara S. and Melo, Hygor P. M. and Furtado, Vasco},
  title = {A global empirical study on how street networks facilitate driving longer distances},
  journal = {Scientific Reports},
  volume = {13},
  pages = {18154},
  year = {2023},
  doi = {10.1038/s41598-023-45236-7}
}
@article{arcaute2015,
  author = {Arcaute, Elsa and Hatna, Erez and Ferguson, Peter and Youn, Hyejin and Johansson, Anders and Batty, Michael},
  title = {Constructing cities, deconstructing scaling laws},
  journal = {Journal of the Royal Society Interface},
  volume = {12},
  pages = {20140745},
  year = {2015},
  doi = {10.1098/rsif.2014.0745}
}
@article{macharia2023,
  author = {Macharia, Peter M. and Banke-Thomas, Aduragbemi and Be\v{n}ov\'a, Lenka},
  title = {Advancing the frontiers of geographic accessibility to healthcare services},
  journal = {Communications Medicine},
  volume = {3},
  pages = {158},
  year = {2023},
  doi = {10.1038/s43856-023-00391-w}
}
"""
(ARTIGO/"referencias.bib").write_text(bibliografia,encoding="utf-8")
print("Bibliografia local escrita:",ARTIGO/"referencias.bib")


Bibliografia local escrita: /home/lfurtado/ciencia-das-cidades/artigo/referencias.bib


## 2. Texto do artigo

O texto abaixo é preenchido diretamente dos CSVs. A redação evita tratar associação como causa, o expoente como prova de localização ótima ou o P90 como limite clínico.


In [3]:
def numero(valor, casas=3):
    return f"{valor:.{casas}f}".replace(".", r"{,}").replace("-", r"\ensuremath{-}")

def intervalo(inf, sup, casas=3):
    return f"[{numero(inf, casas)};\\,{numero(sup, casas)}]"

def milhar(valor):
    return f"{int(round(valor)):,}".replace(",", ".")

def escapar_latex(texto):
    especiais = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",
                 "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}", "~": r"\textasciitilde{}",
                 "^": r"\textasciicircum{}"}
    return "".join(especiais.get(c, c) for c in texto)

s = distancias.loc[distancias.versao == "SUS"]
n_setores_validos = int(s.n_setores.sum())
populacao_valida = int(s.populacao.sum())
habitados = pd.read_csv(RAIZ/"dados/processados/setores_capitais_resumo.csv")
habitados = habitados[habitados.NM_MUN != "Aracaju"]
assert len(habitados) == 26
n_setores_hab, pop_hab = int(habitados.setores_habitados.sum()), int(habitados.populacao.sum())
assert n_setores_validos <= n_setores_hab and populacao_valida <= pop_hab
exemplo = pd.read_csv(RAIZ/"dados/processados/analises/metodo/exemplo_setor_fortaleza.csv").set_index("pa")
assert exemplo.loc["PA1", "d"] < exemplo.loc["PA2", "d"] and exemplo.loc["PA2", "reta"] < exemplo.loc["PA1", "reta"]
cap = cap[cap.n_unidades > 0].copy()
cap["por100k"] = cap.n_unidades / cap.populacao_2022 * 1e5
nome = lambda m: m.split(" - ")[0]
cmin, cmax = cap.loc[cap.por100k.idxmin()], cap.loc[cap.por100k.idxmax()]
sus_c, sus_d = nulo.loc[("SUS", "conjunto")], nulo.loc[("SUS", "dentro da capital")]
agl_s = l1agl.loc[("SUS", "aglomerado urbano")]
gmin, gmax = gini_cap.gini.idxmin(), gini_cap.gini.idxmax()
pv = gini_cap.loc["Porto Velho"]
pv_longe = pd.read_csv(BASE/"distancias/porto_velho_setores_longe.csv").set_index("nome").valor
distritos_pv = pv_longe.distritos_100km_fora_sede.split("; ")[:3]
assert gmax == "Porto Velho"
assert sus_c.alpha_nulo_p2_5 <= l2.loc["SUS", "alpha"] <= sus_c.alpha_nulo_p97_5, "o texto diz que alpha cai na faixa do embaralhamento conjunto"
assert l2.loc["SUS", "alpha"] > sus_d.alpha_nulo_p97_5, "o texto diz que alpha fica acima da faixa dentro das capitais"
assert l1.loc["SUS", "ic95_inferior"] < 1 < l1.loc["SUS", "ic95_superior"] and agl_s.ic95_inferior < 1 < agl_s.ic95_superior
assert num.oeste_pct_renda_q1 > num.resto_pct_renda_q1 and num.oeste_d_media_km < num.resto_d_media_km

template = r"""\documentclass[10pt,a4paper,twocolumn]{article}
\usepackage[brazil]{babel}
\usepackage[utf8]{inputenc}
\usepackage[T1]{fontenc}
\usepackage[margin=2cm,columnsep=0.65cm]{geometry}
\usepackage{graphicx,amsmath,microtype,url}
\usepackage[numbers,sort&compress]{natbib}
\usepackage[font=footnotesize,labelfont=bf]{caption}
\usepackage{titlesec}
\titleformat{\section}{\normalfont\bfseries\normalsize}{\thesection.}{.4em}{}
\titlespacing*{\section}{0pt}{1.1ex plus .2ex}{.5ex}
\setlength{\parindent}{1em}
\setlength{\parskip}{0pt}
\setlength{\textfloatsep}{8pt}
\setlength{\dbltextfloatsep}{8pt}
\setlength{\floatsep}{6pt}
\renewcommand{\dbltopfraction}{0.9}
\setcounter{dbltopnumber}{4}
\renewcommand{\dblfloatpagefraction}{0.8}
\makeatletter
\setlength{\@fptop}{0pt}
\setlength{\@dblfptop}{0pt}
\makeatother
\setlength{\bibsep}{1pt}
\title{\vspace{-1.4em}\large\bfseries Os pronto-atendimentos das capitais brasileiras est\~ao bem situados? Quantidade, densidade, renda e desigualdade das dist\^ancias}
\author{\normalsize @AUTOR@\\[-1pt]\small @AFILIACAO@}
\date{}
\begin{document}
\maketitle
\vspace{-1.2em}

\section{Introdução}
\textbf{Problema.} Em uma urgência de saúde, a distância que o paciente precisa percorrer até o atendimento pode importar. Nas capitais brasileiras, parte desse atendimento é feita por pronto-atendimentos públicos. Contar unidades, porém, não diz quem mora perto delas: duas cidades com a mesma oferta por habitante podem ter distâncias muito diferentes.

Chamamos de pronto-atendimento (PA) o estabelecimento que atende urgências sem hora marcada e está cadastrado como tipo 73 no Cadastro Nacional de Estabelecimentos de Saúde (CNES). O nome muda de cidade para cidade: a Unidade de Pronto Atendimento (UPA) é a mais comum; a Assistência Médica Ambulatorial (AMA) é o nome usado em São Paulo; e o Serviço de Pronto Atendimento (SPA) é o nome usado em Manaus.

\textbf{Objetivo.} Avaliar se os PAs das capitais estão bem situados: primeiro comparando as capitais entre si e depois estimando quanto cada morador percorre pelas ruas até o PA mais próximo. Dividimos esse objetivo em três perguntas norteadoras, que dão nome às partes dos Métodos e dos Resultados:
\begin{enumerate}\itemsep0pt
\item[P1.] \textbf{Quantidade.} O número de PAs de uma capital cresce na mesma proporção que a sua população?
\item[P2.] \textbf{Densidade.} Dentro das capitais, áreas com mais moradores por km$^2$ têm também mais PAs por km$^2$?
\item[P3.] \textbf{Desigualdade dentro da capital: quanto e quem.} Na mesma capital, os moradores ficam a distâncias parecidas do PA mais próximo, ou uma minoria fica muito mais longe? E quem fica mais longe: moradores de setores mais pobres, menos densos ou ambos?
\end{enumerate}
P1 e P2 usam relações de escala: em $Y\propto P^{\beta}$, $\beta=1$ significa crescimento proporcional \citep{bettencourt2021}, e o limite usado para definir a cidade pode alterar o expoente \citep{arcaute2015}. Para P2, há uma referência teórica: se os PAs fossem posicionados para minimizar a distância média dos moradores até o mais próximo, a densidade de PAs cresceria com a densidade de moradores elevada a $2/3$ ($D\propto\rho^{2/3}$) \citep{gastner2006}. Usamos esse valor só como comparação, não como meta. P3 usa a distância de cada setor ao PA mais próximo pelas ruas, que mostra quanto cada morador precisaria percorrer, algo que as contagens de P1 e P2 não mostram.

\section{Métodos}
\textbf{Dados.} (i) CNES/OpenDataSUS (24/10/2026): lista e posição dos @N_SUS@ PAs que atendem o Sistema Único de Saúde (SUS), com coordenadas conferidas, em @N_CAP@ capitais. Pelo nome, @N_UPA@ são UPAs, @N_AMA@ AMAs (São Paulo), @N_SPA@ SPAs (Manaus) e @N_OUTROS@ têm outros nomes. Aracaju não tem estabelecimento cadastrado como tipo 73 (suas unidades municipais de urgência constam como hospital) e fica fora das análises. (ii) Censo 2022 do Instituto Brasileiro de Geografia e Estatística (IBGE): população, área em km$^2$ e renda média do responsável pelo domicílio por setor censitário. (iii) OpenStreetMap: ruas dirigíveis de cada capital.

\begin{figure}[!t]
\centering
\includegraphics[width=0.9\columnwidth]{../figuras/metodo/metodo_setor_pa.pdf}
\caption{Do setor ao PA, em Fortaleza. (a) Ampliação: um setor (quadrado preto), sua ligação $a$ ao cruzamento mais próximo (pontilhado) e o menor percurso pelas ruas até o PA 1 (azul) e até o PA 2 (laranja). Pelas ruas o PA 1 é o mais próximo, embora em linha reta o PA 2 esteja mais perto. O fundo colorido indica a área de atendimento de cada PA. (b) Cidade inteira: cada cor agrupa os setores atribuídos ao mesmo PA mais próximo pelas ruas; linhas brancas são vias principais, estrelas são PAs do SUS e setores excluídos ficam em cinza. Fontes: CNES/OpenDataSUS, Censo 2022/IBGE e OpenStreetMap.}
\label{fig:metodo}
\end{figure}

\textbf{Distância do setor ao PA (Fig.~\ref{fig:metodo}).} O setor censitário, menor que um bairro, permite relacionar localização, moradores e renda do Censo (média de @MEDIA_SETOR@ moradores por setor neste recorte). Representamos as ruas como uma rede: cada cruzamento (ou ponta de rua sem saída) é um nó, e cada trecho de rua entre dois nós é uma aresta, com comprimento e sentido de circulação. Cada setor $s$ é representado por um ponto dentro de sua área, ligado em linha reta ao cruzamento mais próximo, a uma distância $a_s$; da mesma forma, cada PA $p$ é ligado ao seu cruzamento mais próximo, a uma distância $c_p$. Para cada par setor--PA, o algoritmo de Dijkstra calcula o menor caminho permitido entre os nós, de comprimento $b_{sp}$. A distância atribuída ao setor é $d_s=\min_p(a_s+b_{sp}+c_p)$: escolhemos o PA com o menor percurso total, incluindo as duas ligações à rede. Na Fig.~\ref{fig:metodo}a, um setor de Fortaleza fica a $a_s=@EX_A1@$ m do cruzamento mais próximo; pelas ruas, o PA 1 está a $@EX_A1@+@EX_B1@+@EX_C1@=@EX_D1@$ m e o PA 2 a $@EX_A2@+@EX_B2@+@EX_C2@=@EX_D2@$ m. O setor é atribuído ao PA 1, embora em linha reta o PA 2 esteja mais perto (@EX_RETA2@ contra @EX_RETA1@ m). Essa distância é estimada, não é tempo de viagem nem indica qual PA o morador realmente procurou. Setores atribuídos ao mesmo PA formam sua \emph{área de atendimento} (Fig.~\ref{fig:metodo}b), não uma área oficial do SUS. Das @N_CAP@ capitais, partimos de @NSET_HAB@ setores com moradores (@POP_HAB@ moradores) e excluímos os sem rota até um PA ou a mais de 2 km do cruzamento mais próximo ($a_s>2$ km); restam @NSETORES@ setores e @POPULACAO@ moradores (@PCT_POP_VALIDA@\% da população).

\textbf{P1. Quantidade (L1, entre capitais).} Ajustamos por mínimos quadrados $\log_{10}N_i=c+\beta\log_{10}P_i$, em que $N_i$ é o número de PAs e $P_i$ a população da capital $i$. O expoente $\beta$ resume a relação: $\beta=1$ é a relação linear, em que dobrar $P$ dobra $N$ e o número de PAs por habitante é o mesmo em capitais pequenas e grandes; $\beta<1$ (sublinear) indicaria menos PAs por habitante nas capitais maiores, e $\beta>1$ (superlinear), mais. Em geral, dobrar $P$ multiplica $N$ por $2^{\beta}$. Os dados de cada capital são fixos; para medir quanto $\beta$ depende de quais capitais estão na amostra, sorteamos 2.000 vezes um novo conjunto de capitais com reposição (uma capital pode entrar duas vezes e outra ficar de fora) e refazemos o ajuste. O intervalo de confiança de 95\% (IC) vai do 2,5º ao 97,5º percentil desses $\beta$ (semente fixa, para o resultado se repetir). Como o limite municipal é arbitrário, repetimos o ajuste trocando o município da capital pelo seu \emph{aglomerado urbano}: em vez de contar PAs e moradores só na capital, somamos também os municípios vizinhos que têm ao menos 50\% da população na mesma concentração urbana do IBGE.

\textbf{P2. Densidade (L2, áreas dentro das capitais).} Para cada PA $j$, somamos população $P_j$ e área $A_j$ dos setores de sua área de atendimento e contamos os PAs $m_j$ (quase sempre 1). Com as densidades $\rho_j=P_j/A_j$ (moradores) e $D_j=m_j/A_j$ (PAs), ajustamos $\log_{10}D_j=c+\alpha\log_{10}\rho_j$ no conjunto das áreas. Uma área com o dobro de moradores por km$^2$ tem $2^{\alpha}$ vezes mais PAs por km$^2$. Com $\alpha=1$, o número de moradores por PA é o mesmo em áreas densas e esparsas; com $\alpha<1$, cada PA atende mais moradores nas áreas mais densas. O IC vem de 2.000 reamostragens das áreas, sem corrigir dependência espacial. \emph{Controle por embaralhamento.} Quase toda área tem um só PA, então $D_j\approx1/A_j$; e $\rho_j=P_j/A_j$ também divide por $A_j$. Assim, uma área pequena tem, ao mesmo tempo, $D$ alto e $\rho$ alto, e $\alpha$ sai positivo mesmo que a população de cada área fosse sorteada. Para saber quanto de $\alpha$ vem só dessa divisão, fazemos um teste: cada área mantém seu tamanho e seus PAs, mas recebe a população de outra área, escolhida ao acaso; recalculamos $\alpha$ e repetimos 5.000 vezes (semente 42), trocando entre todas as áreas e, separadamente, só entre áreas da mesma capital. Esses sorteios dão o $\alpha$ esperado quando a população não tem relação com a localização dos PAs. Se o $\alpha$ observado ficar entre os percentis 2,5 e 97,5 dos sorteios, a inclinação é explicada pela divisão pela área e não mostra que os PAs se concentram onde os moradores se concentram.

\textbf{P3. Desigualdade dentro da capital: quanto e quem.} \emph{Quanto (Gini da distância).} Cada morador recebe a distância do seu setor até o PA mais próximo. Em cada capital, ordenamos os moradores do mais perto ao mais longe; a curva de Lorenz compara a fração acumulada de moradores com a fração acumulada da soma de suas distâncias. Se todos os moradores estivessem à mesma distância do PA (seja ela curta ou longa), a curva seria a diagonal e o Gini $G$ valeria 0; $G$ cresce quando uma minoria tem percursos muito maiores que os demais. O Gini responde, portanto, se a distância até o PA varia muito de um morador para outro da mesma capital; não mede se a distância é grande: uma capital em que todos estão a 2 km e outra em que todos estão a 10 km têm, ambas, $G=0$. \emph{Quem (renda e densidade).} Para comparar cidades de tamanhos distintos, usamos a distância relativa $d^{rel}_s=d_s/\bar d_c$, em que $\bar d_c$ é a distância média dos moradores da capital $c$ até o PA mais próximo: a média das distâncias $d_s$ dos setores, cada setor pesando pelo seu número de moradores $P_s$, $\bar d_c=\sum_s P_s d_s/\sum_s P_s$. Por exemplo, numa capital com três setores de 1.000, 2.000 e 1.000 moradores, a 1; 1,5 e 4 km do PA, a média ponderada é $\bar d_c=(1.000\times1+2.000\times1{,}5+1.000\times4)/4.000=2$ km, e o setor a 4 km tem $d^{rel}_s=2$, o dobro da média; $d^{rel}_s=1$ é a média da capital. Dentro de cada capital, separamos os setores em quartis de renda e de densidade ($P_s/A_s$): Q1 reúne os 25\% de setores com menores valores e Q4 os 25\% com maiores; o cruzamento forma 16 grupos. O P90 é a distância até a qual estão 90\% dos moradores da capital; os outros 10\% moram em setores a essa distância ou mais. Para cada grupo, calculamos a parcela dos seus moradores em setores com $d_s\ge$ P90. Se o grupo não fosse diferente do resto da capital, essa parcela seria de cerca de 10\%; acima disso, o grupo fica mais longe que o conjunto. Na síntese, reunimos os setores das @N_CAP@ capitais, cada um comparado ao P90 da sua capital, e calculamos também a distância relativa média; capitais mais populosas pesam mais. Para ver o padrão no mapa, detalhamos Fortaleza, cidade da autora.

\begin{figure*}[!t]
\centering
\includegraphics[width=\textwidth]{../figuras/escala/escala_L1_L2.pdf}
\caption{P1 e P2: escala dos PAs do SUS. (a) PAs por 100 mil habitantes; capitais ordenadas da mais para a menos populosa; em destaque, a maior (Porto Velho) e a menor oferta (Boa Vista). (b) P1: PAs versus popula\c{c}\~ao; reta = ajuste, faixa azul = IC 95\%, tracejado = $\beta=1$. (c) P2: densidades por \emph{\'area de atendimento} de cada PA; tracejados: $\alpha=2/3$ (refer\^encia te\'orica) e $\alpha=1$. (d) Controle do efeito da \'area: $\alpha$ em 5.000 sorteios das popula\c{c}\~oes entre todas as \'areas, mantendo PAs e \'areas fixos (histograma laranja), e $\alpha$ observado (linha azul, com seu IC 95\% em azul claro). Fontes: CNES/OpenDataSUS e Censo 2022/IBGE.}
\label{fig:escala}
\end{figure*}

\section{Resultados}
\textbf{P1. Quantidade.} A oferta mediana é de @PA100K_MED@ PA do SUS por 100 mil habitantes (cerca de um por @HAB_POR_PA@ mil moradores), mas vai de @PA100K_MIN@ em @CAP_MIN@ a @PA100K_MAX@ em @CAP_MAX@ (Fig.~\ref{fig:escala}a). Entre capitais, $\beta=@BETA_SUS@$ (IC 95\% @IC_BETA_SUS@; Fig.~\ref{fig:escala}b): o dobro da população corresponderia a @FATOR_BETA@ vez mais PAs, em vez de duas. O IC é largo porque a oferta por habitante varia muito entre capitais de tamanho parecido (Fig.~\ref{fig:escala}a): conforme as capitais sorteadas, $\beta$ muda bastante. Como o IC inclui 1, não há evidência de que as capitais maiores tenham menos PAs por habitante. Com aglomerados urbanos, $\beta=@BETA_AGL_SUS@$ (@IC_BETA_AGL_SUS@), ainda mais próximo da proporcionalidade.

\textbf{P2. Densidade.} Entre as áreas de atendimento, $\alpha=@ALPHA_SUS@$ (IC 95\% @IC_ALPHA_SUS@; Fig.~\ref{fig:escala}c): uma área com o dobro de moradores por km$^2$ tem, em média, @FATOR_ALPHA@ vez mais PAs por km$^2$, acima da referência $2/3$. Esse valor, sozinho, não mostra que há mais PAs por km$^2$ onde há mais moradores por km$^2$. Quando sorteamos as populações entre as áreas, o que apaga qualquer relação entre onde moram as pessoas e onde estão os PAs, o ajuste ainda dá $\alpha=@NULO_CONJ_MEDIA@$ (@NULO_CONJ_IC@) entre todas as áreas (Fig.~\ref{fig:escala}d) e @NULO_DENTRO_MEDIA@ (@NULO_DENTRO_IC@) dentro de cada capital. Ou seja, um $\alpha$ perto de 0,8 aparece só porque as duas densidades dividem pela mesma área. O que mostra se os PAs se concentram onde os moradores se concentram é a diferença entre o observado e os sorteios: entre todas as áreas, o observado fica dentro da faixa dos sorteios, sem diferença; dentro de cada capital, fica um pouco acima dela, um sinal pequeno de que, numa mesma cidade, áreas mais densas têm PAs mais próximos entre si. Como até os sorteios superam $2/3$, a comparação com o valor teórico também não permite dizer se a localização é ótima ou inadequada.

\begin{figure}[!t]
\centering
\includegraphics[width=0.8\columnwidth]{../figuras/gini/gini_PA.pdf}
\caption{P3: desigualdade das dist\^ancias at\'e o PA do SUS. (a) Moradores ordenados pela dist\^ancia de seu setor ao PA mais pr\'oximo (escala log), em Belo Horizonte (menor Gini) e Porto Velho (maior Gini). (b) Curvas de Lorenz das mesmas capitais; a diagonal representa todos os moradores \`a mesma dist\^ancia do PA. Fontes: CNES/OpenDataSUS, Censo 2022/IBGE e OpenStreetMap.}
\label{fig:gini}
\end{figure}

\begin{figure*}[!t]
\centering
\includegraphics[width=\textwidth]{../figuras/final/figura_principal.pdf}
\caption{P3: quem fica mais longe do PA do SUS. (a) Setores das @N_CAP@ capitais em 16 grupos de quartis de renda e densidade, definidos dentro de cada capital, e as m\'edias de cada linha e coluna (``Todos''). Cada c\'elula mostra a parcela dos moradores do grupo em setores a partir do P90 de sua capital; sem diferen\c{c}a entre grupos, seria cerca de 10\% (azul: menos; laranja: mais). A borda preta destaca os setores mais pobres e menos densos. (b) Fortaleza: moradores por dist\^ancia pelas ruas at\'e o PA do SUS mais pr\'oximo; a linha tracejada \'e o P90: 90\% dos moradores est\~ao abaixo dele (azul) e 10\% a partir dele (laranja). (c) Fortaleza: quartil de renda de cada setor, PAs do SUS (estrelas) e contorno laranja dos setores a partir do P90 (@P90_FORT_KM@ km); a elipse marca a ponta oeste. Fontes: CNES/OpenDataSUS, Censo 2022/IBGE e OpenStreetMap.}
\label{fig:principal}
\end{figure*}

\textbf{P3. Desigualdade dentro da capital: quanto e quem.} \emph{Quanto.} A Fig.~\ref{fig:gini}a compara as capitais de menor e de maior Gini, @GMIN_CAP@ e Porto Velho. Para a maioria dos moradores, as curvas de distância quase coincidem: metade dos moradores de Porto Velho fica a até @PV_MEDIANA@ km do PA mais próximo. A diferença está na cauda: @PV_PCT20@\% dos moradores de Porto Velho (@PV_MOR20@, em @PV_N20@ setores) ficam a mais de 20 km, o que eleva a média para @PV_MEDIA@ km. Para @PV_MOR100@ deles, o percurso passa de 100 km e chega a 262 km; são setores de distritos afastados da sede municipal, como @PV_DISTRITOS@. A distância indica o PA tipo 73 mais próximo pelas ruas; não diz se há outro tipo de atendimento no local. A curva de Lorenz (Fig.~\ref{fig:gini}b) resume essa diferença num número: o Gini vai de @GINI_MIN@ em @GMIN_CAP@ a @GINI_MAX@ em Porto Velho, com mediana @GINI_MED@ entre as @N_CAP@ capitais.

\emph{Quem.} Na matriz (Fig.~\ref{fig:principal}a), cada quadro mostra, para um grupo de setores das @N_CAP@ capitais, a parcela dos seus moradores a partir do P90 da própria capital. Se renda e densidade não importassem, todo quadro teria cerca de 10\%. A coluna ``Todos'' olha só a densidade: a parcela cai de @P90_DENS_Q1@\% nos setores menos densos para @P90_DENS_Q4@\% nos mais densos. A linha ``Todos'' olha só a renda: @P90_R1@\%, @P90_R2@\%, @P90_R3@\% e @P90_R4@\% de Q1 a Q4, sem ordem simples; os mais pobres e os mais ricos ficam pouco acima de 10\%. O quadro com borda preta cruza os dois extremos: nos setores ao mesmo tempo mais pobres e menos densos, onde vivem @MIL_POBRE_RALO@ milhão de pessoas, @POBRE_P90@\% dos moradores estão a partir do P90, cerca de @VEZES_ESP@ vezes o esperado. É mais do que cada fator sozinho produz (@P90_R1@\% entre os mais pobres e @P90_DENS_Q1@\% entre os menos densos): é a combinação de baixa renda e baixa densidade que mais afasta os moradores do PA. Nos setores pouco densos e ricos, a parcela também passa do esperado (@P90_RALO_RICO@\%), mas fica bem abaixo. A distância relativa mostra o mesmo padrão: os setores menos densos ficam a @DREL_DENS_Q1@ vez a média da capital, os mais densos a @DREL_DENS_Q4@, e os pobres e pouco densos a @POBRE_RALO@.

Fortaleza mostra que esse padrão das capitais reunidas não vale para toda cidade. Lá, 90\% dos moradores estão a menos de @P90_FORT_KM@ km do PA mais próximo pelas ruas; esse é o P90 da cidade, e os outros 10\% estão a essa distância ou mais (barras laranja na Fig.~\ref{fig:principal}b). No mapa (Fig.~\ref{fig:principal}c), o contorno laranja cerca esses setores, e eles ficam sobretudo em verde escuro, isto é, em setores de renda alta: @FORT_P90_R4@\% dos moradores do quartil mais rico estão a partir do P90, contra @FORT_P90_R1@\% do mais pobre, e @P90_FORT_Q4@\% de todos os moradores a partir do P90 vivem no quartil mais rico. A elipse tracejada marca a ponta oeste, onde ficam @OESTE_PAS@ dos @FORT_PAS@ PAs do SUS da cidade. Essa região é relativamente mais pobre (@OESTE_Q1@\% dos moradores no quartil mais pobre, contra @RESTO_Q1@\% no restante); seus PAs atendem @OESTE_MORADORES@\% dos moradores em @OESTE_AREA@\% da área, e, com tantos PAs próximos, a distância média é de @OESTE_D@ km, contra @RESTO_D@ km no restante. Em Fortaleza, portanto, a região mais pobre é a mais bem servida, e quem fica mais longe mora sobretudo em áreas ricas. Uma explicação possível, não testada aqui, é que moradores de renda alta usam mais a rede privada.

\section{Conclusão e limitações}
\textbf{Respostas.} \emph{P1:} o número de PAs cresce aproximadamente na proporção da população, sobretudo com aglomerados urbanos. \emph{P2:} áreas mais densas têm mais PAs por km$^2$, mas quase toda essa relação se repete com populações sorteadas; o expoente não permite julgar a localização. \emph{P3, quanto:} a desigualdade das distâncias dentro da capital varia muito (Gini de @GINI_MIN@ a @GINI_MAX@) e é puxada por uma minoria muito distante, não pela maioria: em Porto Velho, metade dos moradores está a até @PV_MEDIANA@ km do PA, mas os distritos afastados ficam a mais de 100 km. \emph{P3, quem:} nas capitais reunidas, o fator que mais pesa é a densidade: @P90_DENS_Q1@\% dos moradores dos setores menos densos estão a partir do P90 da sua capital, contra @P90_DENS_Q4@\% nos mais densos. A renda sozinha não segue uma ordem: os mais pobres e os mais ricos ficam ambos pouco acima dos 10\% esperados. Mas, quando baixa renda e baixa densidade se juntam, a distância cresce: nos setores mais pobres e menos densos, @POBRE_P90@\% dos moradores estão a partir do P90, cerca de @VEZES_ESP@ vezes o esperado. Esse padrão das capitais reunidas não se repete em toda cidade: em Fortaleza, @P90_FORT_Q4@\% dos moradores a partir do P90 vivem em setores do quartil mais rico, enquanto a ponta oeste, mais pobre, concentra PAs.

\textbf{Pergunta do título.} A contagem sugere oferta proporcional à população, mas não garante que os PAs estejam bem situados para todos: há diferenças de distância entre bairros que essa contagem oculta. Não demonstramos causalidade, nem definimos um padrão de ``boa localização'' ou onde instalar outra unidade.

Limitações: cada setor é reduzido a um ponto, e as áreas de atendimento são aproximações, não regiões oficiais; L2 divide ambas as densidades pela mesma área e seus intervalos não modelam dependência espacial; o embaralhamento estático não informa se os PAs precederam mudanças na população; a rede do OpenStreetMap pode ter lacunas, e o corte de 2 km exclui 1.178 moradores; a distância não mede tempo, fila, capacidade ou qualidade do serviço, nem considera outros tipos de unidade; a renda do responsável não mede necessidade clínica; as associações entre @N_CAP@ capitais dependem do recorte territorial.

{\footnotesize
\bibliographystyle{unsrtnat}
\bibliography{referencias}
}
\end{document}
"""
cap_todos = pd.read_csv(BASE/"L1/capitais_todos.csv")
_pa = pd.read_csv(RAIZ/"dados/processados/pa_capitais_corrigido.csv")
_pa = _pa[_pa.atende_sus.astype(bool)]
_nome = _pa.NO_FANTASIA.str.upper()
_tipo = {k: int(_nome.str.contains(r"\b" + k + r"\b").sum()) for k in ["UPA", "AMA", "SPA"]}
_sus = distancias[distancias.versao == "SUS"]
assert l2.loc["SUS", "ic95_inferior"] > 2/3, "o texto diz que alpha fica acima de 2/3"
assert nulo.alpha_nulo_p2_5.min() > 2/3, "o texto diz que o alpha ao acaso fica acima de 2/3"

substituicoes = {
    "@AUTOR@": escapar_latex(AUTOR), "@AFILIACAO@": escapar_latex(AFILIACAO),
    **{f"@EX_{k.upper()}{i}@": milhar(exemplo.loc[f"PA{i}", k]) for i in (1, 2) for k in ["a", "b", "c", "d", "reta"]},
    "@NSETORES@": f"{n_setores_validos:,}".replace(",", r"\,"), "@POPULACAO@": f"{populacao_valida:,}".replace(",", r"\,"), "@NSET_HAB@": f"{n_setores_hab:,}".replace(",", r"\,"), "@POP_HAB@": f"{pop_hab:,}".replace(",", r"\,"), "@PCT_POP_VALIDA@": numero(100*populacao_valida/pop_hab, 2),
    "@N_SUS@": f"{int(cap.n_unidades.sum())}", "@N_CAP@": f"{len(cap)}",
    "@N_UPA@": str(_tipo["UPA"]), "@N_AMA@": str(_tipo["AMA"]), "@N_SPA@": str(_tipo["SPA"]),
    "@N_OUTROS@": str(len(_pa) - sum(_tipo.values())),
    "@MEDIA_SETOR@": str(int(round(_sus.populacao.sum() / _sus.n_setores.sum()))),
    "@PA100K_MED@": numero(cap.por100k.median(), 2), "@PA100K_MIN@": numero(cmin.por100k, 2), "@PA100K_MAX@": numero(cmax.por100k, 2),
    "@CAP_MIN@": escapar_latex(nome(cmin.municipio)), "@CAP_MAX@": escapar_latex(nome(cmax.municipio)),
    "@HAB_POR_PA@": str(int(round(100 / cap.por100k.median()))),
    "@BETA_SUS@": numero(l1.loc["SUS", "beta"]),
    "@IC_BETA_SUS@": intervalo(l1.loc["SUS", "ic95_inferior"], l1.loc["SUS", "ic95_superior"]),
    "@FATOR_BETA@": numero(2**l1.loc["SUS", "beta"], 2),
    "@BETA_AGL_SUS@": numero(agl_s.beta), "@IC_BETA_AGL_SUS@": intervalo(agl_s.ic95_inferior, agl_s.ic95_superior),
    "@ALPHA_SUS@": numero(l2.loc["SUS", "alpha"]),
    "@IC_ALPHA_SUS@": intervalo(l2.loc["SUS", "ic95_inferior"], l2.loc["SUS", "ic95_superior"]),
    "@FATOR_ALPHA@": numero(2**l2.loc["SUS", "alpha"], 2),
    "@NULO_CONJ_MEDIA@": numero(sus_c.alpha_nulo_media, 2), "@NULO_CONJ_IC@": intervalo(sus_c.alpha_nulo_p2_5, sus_c.alpha_nulo_p97_5, 2),
    "@NULO_DENTRO_MEDIA@": numero(sus_d.alpha_nulo_media, 2), "@NULO_DENTRO_IC@": intervalo(sus_d.alpha_nulo_p2_5, sus_d.alpha_nulo_p97_5, 2),
    "@DREL_DENS_Q1@": numero(num.d_rel_dens_Q1, 2), "@DREL_DENS_Q4@": numero(num.d_rel_dens_Q4, 2),
    "@DREL_R1@": numero(num.d_rel_renda_Q1, 2), "@DREL_R2@": numero(num.d_rel_renda_Q2, 2),
    "@DREL_R3@": numero(num.d_rel_renda_Q3, 2), "@DREL_R4@": numero(num.d_rel_renda_Q4, 2),
    "@POBRE_RALO@": numero(num.d_rel_pobre_ralo, 2), "@POBRE_P90@": f"{num.pct_p90_pobre_ralo:.0f}",
    **{f"@P90_DENS_Q{q}@": f"{num[f'pct_p90_dens_Q{q}']:.0f}" for q in [1, 4]},
    **{f"@P90_R{q}@": f"{num[f'pct_p90_renda_Q{q}']:.0f}" for q in [1, 2, 3, 4]},
    "@P90_RALO_RICO@": f"{num.pct_p90_ralo_rico:.0f}", "@MIL_POBRE_RALO@": numero(num.mil_moradores_pobre_ralo/1000, 1),
    "@FORT_P90_R1@": f"{num.fortaleza_pct_p90_renda_Q1:.0f}", "@FORT_P90_R4@": f"{num.fortaleza_pct_p90_renda_Q4:.0f}",
    "@VEZES_ESP@": numero(num.pct_p90_pobre_ralo/10, 0),
    "@P90_FORT_KM@": numero(num.fortaleza_p90_km, 1), "@P90_FORT_Q4@": f"{num.fortaleza_pct_p90_em_renda_q4:.0f}",
    "@OESTE_PAS@": f"{num.oeste_pas:.0f}", "@FORT_PAS@": f"{num.fortaleza_pas:.0f}",
    "@OESTE_MORADORES@": f"{num.oeste_pct_moradores:.0f}", "@OESTE_AREA@": f"{num.oeste_pct_area:.0f}",
    "@OESTE_D@": numero(num.oeste_d_media_km, 1), "@RESTO_D@": numero(num.resto_d_media_km, 1),
    "@OESTE_Q1@": f"{num.oeste_pct_renda_q1:.0f}", "@RESTO_Q1@": f"{num.resto_pct_renda_q1:.0f}",
    "@GINI_MED@": numero(gini_cap.gini.median(), 2), "@GINI_MIN@": numero(gini_cap.gini.min(), 2), "@GINI_MAX@": numero(gini_cap.gini.max(), 2),
    "@GMIN_CAP@": escapar_latex(gmin),
    "@PV_MEDIANA@": numero(s.loc[s.capital == "Porto Velho - RO", "dist_mediana_km"].iloc[0], 1),
    "@PV_MEDIA@": numero(pv.dist_media_km, 1),
    "@PV_N20@": f"{int(float(pv_longe.setores_20km))}", "@PV_MOR20@": milhar(float(pv_longe.moradores_20km)),
    "@PV_PCT20@": numero(float(pv_longe.pct_moradores_20km), 1), "@PV_MOR100@": milhar(float(pv_longe.moradores_100km_fora_sede)),
    "@PV_DISTRITOS@": escapar_latex(", ".join(distritos_pv[:-1]) + " e " + distritos_pv[-1]),
}
texto = template
for marcador, valor in substituicoes.items():
    texto = texto.replace(marcador, valor)
assert not re.search(r"@[A-Z_0-9]+@", texto), re.findall(r"@[A-Z_0-9]+@", texto)
(ARTIGO/"artigo.tex").write_text(texto, encoding="utf-8")
(ARTIGO/"estado_rascunho.json").write_text(
    json.dumps({"autor": AUTOR, "afiliacao": AFILIACAO, "pendencias": ["compilar e conferir o limite de 3 páginas (LaTeX não instalado neste ambiente)"] if not TEM_LATEX else []},
               ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
print("Texto gerado:", ARTIGO/"artigo.tex")
palavras = len(re.sub(r"\\[a-zA-Z]+\*?(\[[^\]]*\])?|[{}$\\]", " ", texto.split(r"\begin{document}")[1]).split())
print("Palavras aproximadas (com legendas):", palavras)

Texto gerado: /home/lfurtado/ciencia-das-cidades/artigo/artigo.tex
Palavras aproximadas (com legendas): 3443


## 3. Compilar e conferir o limite de páginas

Se `latexmk` estiver instalado, compilamos e conferimos o número de páginas, citações não resolvidas e caixas largas. O limite aceito pelo usuário é de **cinco páginas**, mas o manual/documentação indica **três páginas**; se o PDF passar de três, reconfirmar com o professor. Sem LaTeX, esta etapa é pulada e o limite de cinco páginas **fica sem verificação**: compile o `artigo.tex` (por exemplo no Overleaf), confira as páginas e reconfirme a divergência com o professor se necessário.


In [4]:
if not TEM_LATEX:
    print("LaTeX não instalado: etapa de compilação pulada. Limite de 5 páginas NÃO verificado; manual/documentação indica 3, reconfirmar com o professor se ultrapassar 3.")
    (ARTIGO/"verificacao.json").write_text(json.dumps({
        "paginas": None, "limite_paginas": 5, "compilado": False,
        "observacao": "LaTeX não disponível; compilar manualmente e conferir 5 páginas. Manual/documentação indica 3; reconfirmar com o professor se ultrapassar 3."
    }, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
else:
    processo = subprocess.run(
        ["latexmk", "-pdf", "-interaction=nonstopmode", "-halt-on-error", "artigo.tex"],
        cwd=ARTIGO, capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=180
    )
    (ARTIGO/"compilacao.log").write_text(processo.stdout+"\n"+processo.stderr, encoding="utf-8")
    assert processo.returncode == 0, processo.stdout[-5000:]+processo.stderr[-2000:]
    info = subprocess.check_output(["pdfinfo", str(ARTIGO/"artigo.pdf")], text=True)
    paginas = int(re.search(r"Pages:\s+(\d+)", info).group(1))
    log = (ARTIGO/"artigo.log").read_text(errors="replace")
    texto_pdf = subprocess.check_output(["pdftotext", str(ARTIGO/"artigo.pdf"), "-"], text=True)
    assert paginas <= 5, f"PDF com {paginas} páginas: revisar o texto."
    if paginas > 3: print(f"ATENÇÃO: {paginas} páginas; o manual/documentação indica 3. Reconfirmar o limite de 5 páginas com o professor.")
    assert "undefined citations" not in log and "There were undefined references" not in log
    assert "??" not in texto_pdf, "Conferir referência não resolvida."
    print("Páginas:", paginas, "(limite: 5)")
    (ARTIGO/"verificacao.json").write_text(json.dumps({
        "paginas": paginas, "limite_paginas": 5, "compilado": True, "referencias_resolvidas": True,
        "avisos_overfull_hbox": log.count("Overfull \\hbox")
    }, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")


ATENÇÃO: 5 páginas; o manual/documentação indica 3. Reconfirmar o limite de 5 páginas com o professor.
Páginas: 5 (limite: 5)


## 4. Conferência final

- Autoria e afiliação: Lia Sucupira, Universidade de Fortaleza.
- Distância pela soma completa (ligação + rota + ligação): aplicada.
- Conferir visualmente as quatro figuras no PDF e o limite aceito pelo usuário (5 páginas). O manual/documentação indica 3 páginas: se o PDF passar de 3, reconfirmar com o professor.
